Hands-On Lab: Vectorizing Text      
● Step 1: Apply TF-IDF to the cleaned text from Day 1 and train a simple classifier as a text baseline.     
● Step 2: Load pre-trained word embeddings and find the nearest neighbors of a few words to see the semantic geometry.      
● Step 3: If the project is text-based, compare a TF-IDF model against the Week 7 LSTM/transformer on the same metric.      
● Step 4: Document which representation fits the project and why, in Markdown.      

#### Load Dataset:

In [1]:
import pandas as pd


Reusing the same dataset as Sprint 2\day4 imdb reviews data !  

In [2]:
path = r"..\Data\IMDB Dataset of 50K Movie Reviews\IMDB Dataset.csv"

df = pd.read_csv(path) # header=None because this dataset does not use a normal header row

print(df.shape)
df.head()

(50000, 2)


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
print(df.isnull().sum())
print("")
print("Duplicates:", df.duplicated().sum())

review       0
sentiment    0
dtype: int64

Duplicates: 418


In [4]:
df = df.drop_duplicates().reset_index(drop=True)

print("New shape:", df.shape)
print("Duplicates left:", df.duplicated().sum())

New shape: (49582, 2)
Duplicates left: 0


####  Text Preprocessing: (from day1)

In [5]:
import nltk

import string

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
# These are extra language resources NLTK needs.
# (tokenizer data) (downloaded once)
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

##### Function Then applying to DataSet:

In [6]:
# prepare stopwords
stop_words = set(stopwords.words("english"))

stop_words.discard("not")
stop_words.discard("no")
stop_words.discard("nor")

# create a lemmatizer
lemmatizer = WordNetLemmatizer()


# The functions
def preprocess_text(text):
    
    # 1- tokenize
    tokens = word_tokenize(text)
    
    # 2- lowercase
    tokens = [token.lower()
              for token in tokens]
    
    # 3- remove punctuation
    tokens = [token 
              for token in tokens
              if token not in string.punctuation]
    
    # 4- remove stop words
    tokens = [token
              for token in tokens
              if token not in stop_words]
    
    # 5- lemmatize
    tokens = [lemmatizer.lemmatize(token)
              for token in tokens]
    
    # JOIN back into text 
    cleaned_text = " ".join(tokens)
    
    return cleaned_text
    

In [7]:
# Applying to the full dataset
df["cleaned_review"] = df["review"].apply(preprocess_text)

In [8]:
df[ ["review", "cleaned_review", "sentiment"]].head()

,review,cleaned_review,sentiment
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching 1 oz episode '...,positive
1,A wonderful little production. <br /><br />The...,wonderful little production br br filming tech...,positive
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,positive
3,Basically there's a family where a little boy ...,basically 's family little boy jake think 's z...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei 's `` love time money '' visuall...,positive


##### compare few original vs cleaned reviews: 

In [9]:
for i in range(3):
    print(f"--- REVIEW {i+1} ---")

    print("\nRAW:")
    print(df["review"].iloc[i])

    print("\nCLEANED:")
    print(df["cleaned_review"].iloc[i])

    print("\n" + "=" * 80 + "\n")

--- REVIEW 1 ---

RAW:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the

#### `TF-IDF`

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

##### Spliting the preprocessed Data:

In [ ]:
X = df["cleaned_review"]  # using the Cleeaned review data ! 
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,random_state=42,stratify=y)

print("Y train distribution: ",y_train.value_counts())
print("\nY test distribution: ",y_test.value_counts())

Y train distribution:  sentiment
positive    19907
negative    19758
Name: count, dtype: int64

Y test distribution:  sentiment
positive    4977
negative    4940
Name: count, dtype: int64


##### tf-idf model:

TF-IDF does NOT understand meaning.

It understands:     
how important a word isbased on frequency.

In [16]:
# create model
tfidf = TfidfVectorizer(max_features=5000)

In [ ]:
# apply tf-idf

X_train_tfidf = tfidf.fit_transform(X_train) #only fit the training data

X_test_tfidf  =  tfidf.transform(X_test) 

In [18]:
# baseline model to test td-idf

tfidf_model = LogisticRegression(max_iter=1000)

tfidf_model.fit(X_train_tfidf, y_train)

y_pred_tfidf = tfidf_model.predict(X_test_tfidf)

In [20]:
print("Accuracy:",accuracy_score(y_test, y_pred_tfidf))
print("")
print(classification_report( y_test, y_pred_tfidf))

Accuracy: 0.8882726631037612

              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4940
    positive       0.88      0.90      0.89      4977

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917



#### `Pre-trained word embeddings`

we want to demonstrate that `embeddings capture semantic similarity`.

In [21]:
import gensim.downloader as api

In [22]:
# pretrained GloVe embeddings model.

embedding_model = api.load("glove-wiki-gigaword-50") # 50 means each word is represented by a vector with 50 numeric dimensions.

[==================================================] 100.0% 66.0/66.0MB downloaded


##### Finding Neartest neighbors (similarity) for a few words:

In [23]:
embedding_model.most_similar("good", topn=5)

[('better', 0.9284391403198242),
 ('really', 0.9220623970031738),
 ('always', 0.9165270924568176),
 ('sure', 0.903351366519928),
 ('something', 0.9014206528663635)]

In [24]:
embedding_model.most_similar("bad", topn=5)

[('worse', 0.8878378868103027),
 ('unfortunately', 0.8650501370429993),
 ('too', 0.8608258366584778),
 ('really', 0.8486315011978149),
 ('little', 0.8427671194076538)]

In [25]:
embedding_model.most_similar("movie", topn=5)

[('movies', 0.9322481155395508),
 ('film', 0.9310100078582764),
 ('films', 0.8937394618988037),
 ('comedy', 0.8902585506439209),
 ('hollywood', 0.8718216419219971)]

This demonstrates :

similar meaning     
↓       
similar vectors     
↓       
close together mathematically       

In [29]:
# we can also calculate similarity between two words directly.

print ("similarity between 'good' and 'great': ")
print(embedding_model.similarity( "good","great"))

print ("\nsimilarity between 'good' and 'computer': ")
print(embedding_model.similarity("good","computer"))

similarity between 'good' and 'great': 
0.7982692

similarity between 'good' and 'computer': 
0.48246917


**Pre-trained Word Embeddings**

Pre-trained GloVe embeddings were used to explore semantic relationships between words.

The nearest neighbors of words such as good, bad, and movie showed that:     
 words with similar meanings or related usage tend to appear close together in the embedding vector space.

For example, the similarity between good and great was approximately 0.798, while the similarity between good and computer was only about 0.482.

This demonstrates that word embeddings capture semantic relationships between words, unlike TF-IDF, which mainly represents word importance based on frequency.

#### `TF-IDF vs Transformer (from Sprint 2 day4) `

Since we Tested Transformer using a sample of 200,      
 to keep it fair, we are doing the same here.         

In [30]:
X_compare = X_test.iloc[:200]
y_compare = y_test.iloc[:200]

In [ ]:
X_compare_tfidf = tfidf.transform( X_compare) # transform the sample using tfidf

tfidf_compare_pred = tfidf_model.predict(X_compare_tfidf) # then predict(with logistic regression) using it.

In [33]:
print(accuracy_score( y_compare, tfidf_compare_pred))
print (classification_report(y_compare, tfidf_compare_pred))

0.915
              precision    recall  f1-score   support

    negative       0.92      0.91      0.92       105
    positive       0.91      0.92      0.91        95

    accuracy                           0.92       200
   macro avg       0.91      0.92      0.91       200
weighted avg       0.92      0.92      0.92       200



The TF-IDF + Logistic Regression baseline achieved 91.5% accuracy on the same 200 IMDb reviews used for the Transformer evaluation.     
 Performance was balanced across both sentiment classes, with macro F1 around 0.91.     

| Model                        | Representation                    |  Accuracy | Macro Precision | Macro Recall | Macro F1 |
| ---------------------------- | --------------------------------- | --------: | --------------: | -----------: | -------: |
| TF-IDF + Logistic Regression | TF-IDF                            | **0.915** |        **0.91** |     **0.92** | **0.91** |
| DistilBERT                   | Contextual Transformer embeddings |     0.860 |          0.8616 |       0.8581 |   0.8591 |


**TF-IDF vs DistilBERT (transformer) Comparison**

The TF-IDF + Logistic Regression model achieved an accuracy of 91.5% and a macro F1-score of approximately 0.91 on the 200-review evaluation sample.

The pretrained DistilBERT Transformer achieved 86.0% accuracy and a macro F1-score of approximately 0.859 on the same reviews.

`In this experiment`, the TF-IDF baseline performed better.   

 `One reason is` that the Logistic Regression classifier was trained directly on the IMDb training data,      
  while the DistilBERT model was used as a pretrained sentiment classifier without additional fine-tuning on this dataset.      

#### `Representation Choice`

**Which representation fits and why?**

TF-IDF is a strong representation for this sentiment-classification task because it is simple, fast, and produced the best result in the experiment.

TF-IDF represents words according to their importance in a document, but it does not directly capture semantic meaning or context.

Pre-trained GloVe embeddings provide richer semantic information.   

 The nearest-neighbor experiment showed that related words such as movie, film, and movies are  
  close together in vector space, and good was more similar to great than to computer.

Transformer representations such as DistilBERT go further by creating contextual embeddings, where a word's representation depends on the sentence around it.

For this dataset and experiment, TF-IDF + Logistic Regression is the most suitable representation because it achieved the highest accuracy while remaining computationally efficient.   
 However, embeddings and Transformers are more appropriate when deeper semantic meaning and context are important.